### 1. 简单解释：`git add .` 是在干什么？

想象你在要在淘宝上买东西（提交代码）：

1.  **工作区（写代码）**：你在货架上挑挑拣拣，把想买的商品（修改过的文件）拿在手里。
2.  **暂存区（Staging Area）**：`git add .` 相当于**把手里的商品全部放进购物车**。
    *   `.` 代表“当前目录下的所有文件”。
    *   如果你只想买其中一样，可以用 `git add main.cpp`。
3.  **仓库（Repository）**：`git commit` 相当于**去收银台结账**。
    *   结账后，生成一张“小票”（Commit ID），这笔交易就被永久记录下来了。

**总结**：`git add .` 就是**把所有修改过的文件“放入购物车”，准备下一步打包存档。**

---

### 2. 进阶开发：利用 Compute Shader 实现“水波纹仿真”

既然你的红色方块和动态纹理都跑通了，现在的你也掌握了 **GPU 并行计算** 的基础。

接下来的**水波纹（Ripple Effect）** 是一个非常经典的物理仿真案例。它和之前的“画个 `sin(t)` 图案”有本质区别：
*   **之前的**：像素颜色只取决于**时间**和**坐标**（死板的数学公式）。
*   **现在的**：像素颜色取决于**周围像素的状态**和**上一帧的状态**（真实的物理传播）。

#### 核心难点：读写冲突与乒乓技术

在 GPU 中，你不能一边读一张纹理的左边像素，同时修改这张纹理的右边像素。这会导致数据混乱。
**解决方案：乒乓缓冲 (Ping-Pong Buffering)**

你需要 **两张** 一模一样的纹理：
*   **Texture A (上一帧)**：只读 (Input)。
*   **Texture B (当前帧)**：只写 (Output)。
*   **下一帧**：交换它们的身份。

#### 具体的开发步骤

##### 第一步：C++ 端改造 (创建双纹理)

你需要修改你的 `TestComputeShader` 类：

1.  **资源升级**：
    创建两个 `ImageTexture` 指针，比如 `m_TextureIn` 和 `m_TextureOut`。它们的大小必须一样（512x512），格式建议用 `GL_RGBA32F`（因为需要存负数的高度和速度，普通 `RGBA8` 存不了负数）。

2.  **渲染循环 (`OnRender`) 改造**：
    每一帧，你都要指派谁是读，谁是写。

    ```cpp
    // 伪代码逻辑
    void OnRender() {
        // 1. 绑定资源
        // 绑定 TextureIn 到 image unit 0 (作为输入源)
        m_TextureIn->BindImage(0, GL_READ_ONLY); 
        // 绑定 TextureOut 到 image unit 1 (作为输出目标)
        m_TextureOut->BindImage(1, GL_WRITE_ONLY);

        // 2. 传入交互参数 (鼠标点击哪里？)
        m_ComputeShader->Bind();
        m_ComputeShader->SetUniform("u_MousePos", mouseX, mouseY);
        m_ComputeShader->SetUniform("u_MouseDown", isMouseDown);

        // 3. 计算！
        glDispatchCompute(512/8, 512/8, 1);
        glMemoryBarrier(GL_ALL_BARRIER_BITS);

        // 4. 显示
        // 我们要看的是刚刚算好的 TextureOut
        m_Shader->Bind();
        m_TextureOut->Bind(0); 
        RenderScreenQuad();

        // 5. 【关键】交换指针 (Ping-Pong)
        // 下一帧，Out 变 In，In 变 Out
        std::swap(m_TextureIn, m_TextureOut);
    }
    ```

##### 第二步：Shader 端算法 (波动方程)

在 `res/shaders/Wave.compute` 中，你需要实现简单的波动物理。
我们需要存两个核心数据：
*   **高度 (Height)**：水面的高低。存放在 `Red` 通道。
*   **垂直速度 (Velocity)**：水面上下运动的速度。存放在 `Green` 通道。

```glsl
#version 430 core
layout(local_size_x = 8, local_size_y = 8) in;

// 两个纹理接口
layout(rgba32f, binding = 0) uniform image2D img_input;  // 上一帧
layout(rgba32f, binding = 1) uniform image2D img_output; // 当前帧

uniform vec2 u_MousePos; // 鼠标点击位置 (归一化 0~1)
uniform int u_MouseDown; // 鼠标是否按下

void main() {
    ivec2 coord = ivec2(gl_GlobalInvocationID.xy);
    ivec2 size = imageSize(img_input);

    // 1. 读取上一帧的自身状态
    vec4 state = imageLoad(img_input, coord);
    float h = state.r; // 高度
    float v = state.g; // 速度

    // 2. 读取上下左右邻居的高度 (模拟拉力)
    // 注意：实际写代码时要加边界检查 (if coord.x > 0 ...)
    float h_up    = imageLoad(img_input, coord + ivec2(0, 1)).r;
    float h_down  = imageLoad(img_input, coord + ivec2(0, -1)).r;
    float h_left  = imageLoad(img_input, coord + ivec2(-1, 0)).r;
    float h_right = imageLoad(img_input, coord + ivec2(1, 0)).r;

    // 3. 物理计算 (平均值 - 自身 = 拉普拉斯算子)
    // 意思就是：如果你比周围高，就会被拉下去；比周围低，就会被拉上来
    float force = (h_up + h_down + h_left + h_right) / 4.0 - h;

    // 4. 更新速度和位置 (欧拉积分)
    v += force * 0.5; // 力改变速度 (0.5是模拟波速)
    v *= 0.99;        // 阻尼 (0.99让波慢慢停下来，不然会震荡到永远)
    h += v;           // 速度改变高度

    // 5. 处理鼠标交互 (制造波源)
    // 如果鼠标按下了，且当前像素离鼠标很近，就把高度强制提起来
    if (u_MouseDown == 1) {
        float dist = distance(vec2(coord), u_MousePos * vec2(size));
        if (dist < 5.0) {
            h = 1.0; // 只要点这里，高度就是 1
        }
    }

    // 6. 写入下一帧
    // Blue通道用来做显示颜色，看起来像水
    imageStore(img_output, coord, vec4(h, v, h * 0.5 + 0.5, 1.0));
}
```

### 总结

这个练习会让你学到 FEM (有限元) 和 CFD (流体) 最基础的思维模式：
1.  **离散化**：把连续的水面变成 512x512 个格子。
2.  **时间积分**：把时间变成一帧一帧的迭代。
3.  **边界条件**：处理纹理边缘。
4.  **交互**：如何把 CPU 的输入（鼠标）传给 GPU 计算核心。

做完这个，你就是半个图形学模拟工程师了！